In [3]:
import zipfile, json, torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from rouge_score import rouge_scorer
import numpy as np

In [4]:
# Load SAD data (already cloned)
def load_jsonl_from_zip(zip_path, file_name, password=b'sadtimesforthesetimes'):
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        content = zip_ref.read(file_name, pwd=password).decode('utf-8')
        return [json.loads(line) for line in content.strip().split('\n')]

zip_path = './sad/sad/stages/private_data_gen.zip'
deploy = load_jsonl_from_zip(zip_path, 'out/deploy_oversight.jsonl')
test = load_jsonl_from_zip(zip_path, 'out/test_oversight.jsonl')
all_data = deploy + test
print(f"Loaded {len(all_data)} SAD questions")

Loaded 400 SAD questions


In [13]:
model_name = "meta-llama/Llama-3.2-3B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name, torch_dtype=torch.bfloat16)
model.eval()
print("Model loaded")

Loading checkpoint shards: 100%|██████████| 2/2 [00:00<00:00,  5.94it/s]


Model loaded


In [14]:
scorer = rouge_scorer.RougeScorer(['rougeL'], use_stemmer=True)
results = []

for item in all_data[:100]:  # 100 questions is enough
    question = item['body']
    tokens = tokenizer.encode(question)
    
    # Truncate to first 60%
    cutoff = int(len(tokens) * 0.6)
    truncated_tokens = tokens[:cutoff]
    truncated_text = tokenizer.decode(truncated_tokens)
    
    # Generate completion
    inputs = tokenizer(truncated_text, return_tensors="pt")
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=50,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )
    
    completion = tokenizer.decode(outputs[0][inputs['input_ids'].shape[1]:], 
                                   skip_special_tokens=True)
    
    # Score against full original question
    remaining = tokenizer.decode(tokens[cutoff:])
    score = scorer.score(remaining, completion)['rougeL'].fmeasure
    results.append(score)

print(f"Mean ROUGE-L: {np.mean(results):.4f}")
print(f"Std: {np.std(results):.4f}")
print(f"If this is near 0.0-0.1 → no contamination signal")
print(f"If this is near 0.3+ → possible contamination")

/Users/architmanek/miniconda3/envs/repoenv/lib/python3.11/site-packages/transformers/generation/configuration_utils.py:629: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.6` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
/Users/architmanek/miniconda3/envs/repoenv/lib/python3.11/site-packages/transformers/generation/configuration_utils.py:634: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.9` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
  warnings.warn(


Mean ROUGE-L: 0.1206
Std: 0.1050
If this is near 0.0-0.1 → no contamination signal
If this is near 0.3+ → possible contamination
